In [5]:
print("Okay")

Okay


In [6]:
from dotenv import load_dotenv
import os

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [7]:
# Prefer a Google API key from the environment or .env file.
# If it is missing or rate-limited, the notebook will fall back to local embeddings.
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")

In [8]:
# --------------------------------------------------
# 3. Load PDF
# --------------------------------------------------
file_path = r"D:\New_file\llama2-research-paper.pdf"
pdf_loader = PyPDFLoader(file_path)
pdf_data = pdf_loader.load()
print("Total pdf_data:", len(pdf_data))

Total pdf_data: 77


In [9]:
# --------------------------------------------------
# 4. Create chunks
# --------------------------------------------------
chunker = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunked_data = chunker.split_documents(pdf_data)
print("Total chunked_data:", len(chunked_data))

Total chunked_data: 175


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14158.80it/s]


In [13]:
embedding_function=embeddings

In [14]:
# --------------------------------------------------
# 5. Create Chroma vector store
# --------------------------------------------------
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)


In [15]:
# --------------------------------------------------
# 6. Add documents
# --------------------------------------------------

document_ids = vector_store.add_documents(
    documents=chunked_data
)

print("Documents added:", len(document_ids))


Documents added: 175


In [16]:
# --------------------------------------------------
# 7. Create retriever
# --------------------------------------------------

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [17]:
# --------------------------------------------------
# 8. Test retriever
# --------------------------------------------------

query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)


--- Retrieved document 1 ---
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delayi
Metadata: {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'trapped': '/False', 'total_pages': 77, 'page_label': '4', 'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'page': 3, 'creationdate': '2023-07-20T00:30:36+00:00', 'title': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'source': 'D:\\New_file\\llama2-research-paper.pdf', 'subject': '', 'author': ''}

--- Retrieved docu

In [18]:
# --------------------------------------------------
# 9. Prompt
# --------------------------------------------------

prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)


In [19]:
# --------------------------------------------------
# 10. Format documents
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )

In [27]:
# --------------------------------------------------
# 11. LLM
# --------------------------------------------------

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [28]:
# --------------------------------------------------
# 12. RAG chain
# --------------------------------------------------

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [29]:
# --------------------------------------------------
# 13. Ask question
# --------------------------------------------------

answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)

d:\New_file\env2\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Final answer:

Based on the provided context, the architecture of Llama 2 is as follows:

* **Base Architecture:** It is an **auto-regressive language model** that uses an **optimized transformer architecture**.
* **Context Length:** It features an expanded context window of **4,096 tokens** (increased from 2,048 tokens in Llama 1).
* **Attention Mechanism:** It utilizes **Grouped-Query Attention (GQA)**, where key (K) and value (V) projections are shared across multiple heads to reduce memory costs associated with the KV cache size during decoding.
* **Tuned Variations:** The tuned versions incorporate **supervised fine-tuning (SFT)** and **reinforcement learning with human feedback (RLHF)** to align with human preferences for helpfulness and safety.


In [30]:
from langchain_chroma import Chroma

loaded_vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2"
)

In [31]:
loaded_retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [32]:
docs = loaded_retriever.invoke(
    "What is Llama 2?"
)

for doc in docs:
    print(doc.page_content[:500])

guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delayi
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delay

In [33]:
vector_store.persist()

AttributeError: 'Chroma' object has no attribute 'persist'

In [34]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {
            "page": 10
        }
    }
)

In [35]:
results = vector_store.similarity_search(
    query="What is reinforcement learning?",
    k=5,
    filter={
        "page": 10
    }
)